# 00_smoke — ATML-PA4 Colab smoke test

**Run cells top-to-bottom on a Colab L4 runtime.**  
Every code cell prints `PASS` or `FAIL`.  
Re-run any cell after fixing issues; variables persist within the session.

| Cell | What it checks |
|---|---|
| 1 | Drive mount + HF_HOME on Drive |
| 2 | Clone / pull repo, add to sys.path |
| 3 | pip install pinned deps |
| 4 | Write run_config.json (points config at Drive data dir) |
| 5 | CPU unit tests (pytest) |
| 6 | **T1 GPU**: FP16 baseline PPL (5 synthetic windows) + HellaSwag (8 examples) |
| 7 | **T2 GPU**: dense/tiled/SDPA agreement at L=256 + 50 MQAR training updates |
| 8 | **T3 CPU**: chi2 Markov test (correct sampler) + negative control |


In [ ]:
# Cell 1: Drive mount + HF_HOME
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = '/content/drive/MyDrive'
except Exception as _e:
    print(f'Drive not available ({_e}); using $HOME for cache paths')
    DRIVE_ROOT = os.path.expanduser('~')

HF_HOME = os.path.join(DRIVE_ROOT, 'atml_pa4_hf_cache')
DATA_DIR = os.path.join(DRIVE_ROOT, 'atml_pa4_data')  # populated by setup_data.py

os.makedirs(HF_HOME, exist_ok=True)
os.environ['HF_HOME'] = HF_HOME
os.environ['TRANSFORMERS_CACHE'] = HF_HOME
os.environ['HF_DATASETS_CACHE'] = HF_HOME

print(f'HF_HOME  : {HF_HOME}')
print(f'DATA_DIR : {DATA_DIR}')
print(f'Data dir exists: {os.path.exists(DATA_DIR)}')


In [ ]:
# Cell 2: Clone / pull repo and add to sys.path
import subprocess

REPO_URL = 'https://github.com/reemwaheed13/ATML-PA4'
REPO_DIR = '/content/ATML-PA4'

if os.path.isdir(REPO_DIR + '/.git'):
    r = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull'],
        capture_output=True, text=True
    )
    print('git pull:', r.stdout.strip() or r.stderr.strip())
else:
    r = subprocess.run(
        ['git', 'clone', REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('git clone FAILED:', r.stderr)
    else:
        print('git clone OK')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('CWD:', os.getcwd())


In [ ]:
# Cell 3: pip install pinned dependencies
import subprocess, sys

pip = [sys.executable, '-m', 'pip', 'install', '-q']
# Handout-pinned versions
subprocess.run(pip + ['transformers==4.57.1', 'huggingface_hub==0.36.2'], check=True)
# Other deps
subprocess.run(pip + ['peft>=0.14.0', 'scipy>=1.11.0', 'datasets>=2.19.0', 'accelerate>=0.34.0'], check=True)

import transformers, huggingface_hub
print(f'transformers  {transformers.__version__}')
print(f'huggingface_hub {huggingface_hub.__version__}')
print('PASS: pip installs')


In [ ]:
# Cell 4: Write run_config.json pointing at Drive data dir
import json, pathlib

config = {
    'model_id':          'Qwen/Qwen2.5-1.5B',
    'instruct_model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
    'draft_model_id':    'Qwen/Qwen2.5-0.5B-Instruct',
    'data_dir':          DATA_DIR,
    'results_dir':       os.path.join(REPO_DIR, 'results'),
    'seed':              6304,
    'device':            'cuda:0',
}

cfg_path = pathlib.Path(REPO_DIR) / 'configs' / 'run_config.json'
cfg_path.write_text(json.dumps(config, indent=2))
print('Config written to', cfg_path)
print(json.dumps(config, indent=2))
print('PASS: config written')


In [ ]:
# Cell 5: CPU unit tests
import subprocess, sys

r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True, cwd=REPO_DIR
)
print(r.stdout[-5000:])
if r.returncode == 0:
    print('PASS: all CPU unit tests')
else:
    print('FAIL: CPU unit tests')
    print(r.stderr[-1000:])


In [ ]:
# Cell 6: T1 GPU smoke — FP16 baseline PPL (5 synthetic windows) + HellaSwag (8 examples)
# Uses synthetic JSONL records so this cell does not require real data files.
import json, tempfile, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from common.eval_perplexity import evaluate_perplexity
from common.eval_hellaswag import evaluate_hellaswag

DEVICE = 'cuda:0'
MODEL_ID = 'Qwen/Qwen2.5-1.5B'

print(f'Loading {MODEL_ID} (this downloads ~3 GB on first run, then cached) ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map=DEVICE,
    trust_remote_code=True,
)
model.eval()
print(f'  dtype={next(model.parameters()).dtype}, device={DEVICE}')

# ---- PPL smoke: 5 synthetic 512-token windows ----
vocab_size = tokenizer.vocab_size
_rng = torch.Generator()
_rng.manual_seed(6304)
ppl_records = [
    {'token_ids': torch.randint(1, min(vocab_size, 50000), (512,), generator=_rng).tolist()}
    for _ in range(5)
]
with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False) as _f:
    ppl_tmp = _f.name
    for _rec in ppl_records:
        _f.write(json.dumps(_rec) + '\n')

ppl = evaluate_perplexity(model, ppl_tmp, DEVICE)
print(f'\nT1 PPL smoke: N={ppl["N"]}, NLL={ppl["mean_nll"]:.4f}, PPL={ppl["ppl"]:.2f}')
assert ppl['N'] == 5 * 511, f'Expected {5*511} scored tokens, got {ppl["N"]}'
print('PASS: T1 PPL smoke (5 synthetic windows)')

# ---- HellaSwag smoke: 8 synthetic examples ----
hs_records = [
    {
        'order': i,
        'example': {
            'label': i % 4,
            'ctx': 'A person enters the kitchen and',
            'activity_label': 'Cooking',
            'endings': [
                'opens the refrigerator.',
                'sits down on the floor.',
                'reads a newspaper quietly.',
                'turns off the ceiling lights.',
            ],
        },
    }
    for i in range(8)
]
with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False) as _f:
    hs_tmp = _f.name
    for _rec in hs_records:
        _f.write(json.dumps(_rec) + '\n')

hs = evaluate_hellaswag(model, tokenizer, hs_tmp, DEVICE)
print(f'\nT1 HellaSwag smoke: n={hs["n_examples"]}, raw_acc={hs["raw_acc"]:.3f}, acc_norm={hs["acc_norm"]:.3f}')
assert hs['n_examples'] == 8, f'Expected 8 examples, got {hs["n_examples"]}'
print('PASS: T1 HellaSwag smoke (8 synthetic examples)')

# Free GPU memory
del model
import gc; gc.collect(); torch.cuda.empty_cache()
print('\nGPU memory freed.')


In [ ]:
# Cell 7: T2 GPU smoke — attention agreement at L=256 + 50 MQAR training updates
import random, torch
from task2_attention.attention_ops import (
    dense_causal_attention,
    tiled_causal_attention,
    sdpa_attention,
    compare_to_reference,
)
from task2_attention.mqar_train import train_mqar

DEVICE = 'cuda:0'

# ---- Attention agreement at L=256 ----
print('T2: attention agreement (dense / tiled / SDPA) at L=256 ...')
B, H, L, d = 2, 4, 256, 64
res_dt = compare_to_reference(
    dense_causal_attention, tiled_causal_attention,
    B=B, H=H, L=L, d=d, device=DEVICE, dtype=torch.float32, atol=1e-3
)
res_ds = compare_to_reference(
    dense_causal_attention, sdpa_attention,
    B=B, H=H, L=L, d=d, device=DEVICE, dtype=torch.float32, atol=1e-3
)
print(f'  Dense vs Tiled: max_diff={res_dt["max_abs_diff"]:.2e}  within_atol={res_dt["within_atol"]}')
print(f'  Dense vs SDPA:  max_diff={res_ds["max_abs_diff"]:.2e}  within_atol={res_ds["within_atol"]}')
assert res_dt['within_atol'], 'Dense vs Tiled agreement FAILED'
assert res_ds['within_atol'], 'Dense vs SDPA agreement FAILED'
print('PASS: T2 attention agreement')

# ---- MQAR 50-update smoke ----
def _make_smoke_mqar(n, K=4, T=256, seed=6304):
    rng = random.Random(seed)
    records = []
    for _ in range(n):
        tokens = [rng.randint(2, 63) for _ in range(T)]
        keys = rng.sample(range(64, 128), K)
        vals = rng.sample(range(128, 192), K)
        # Key-value pairs at the start of the sequence
        for i, (k, v) in enumerate(zip(keys, vals)):
            tokens[2 * i] = k
            tokens[2 * i + 1] = v
        # Queries near the end
        q_pos = list(range(T - K, T))
        for i, pos in enumerate(q_pos):
            tokens[pos] = keys[i]
        records.append({'tokens': tokens, 'query_positions': q_pos, 'answer_ids': vals})
    return records

smoke_data = _make_smoke_mqar(64)
print('\nT2: MQAR 50-update training smoke (attn_type=dense) ...')
train_result = train_mqar(
    attn_type='dense',
    train_records=smoke_data,
    val_records={'near': [], 'far': [], 'mixed': []},
    seed=6304,
    device=DEVICE,
    n_updates=50,
    batch_size=8,
    result_dir='/tmp/smoke_mqar',
    checkpoint_steps=[],  # skip checkpoint eval and saves
)
# No steps are logged before update 100; curves['step'] will be []
print(f'  Curves keys: {list(train_result["curves"].keys())}')
print('PASS: T2 MQAR 50-update smoke')


In [ ]:
# Cell 8: T3 CPU smoke — chi2 Markov correctness test + negative control
# This cell is CPU-only; no model loading required.
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from task3_specdec.speculative import SpeculativeSampler

print('T3: chi2 Markov test — 100,000 trials, CPU only ...')

# Correct sampler: p-value must NOT be rejected at p < 0.01
res_correct = SpeculativeSampler.run_chi2_test(
    n_symbols=3, seq_len=3, n_trials=100_000, seed=6304, correct=True
)
print(f'  Correct:   p={res_correct["p_value"]:.4f}  chi2={res_correct["chi2_stat"]:.2f}'
      f'  min_expected={res_correct["min_expected_count"]:.1f}')
assert res_correct['min_expected_count'] >= 10, (
    f'Expected counts must be >= 10; got {res_correct["min_expected_count"]:.1f}'
)
assert not res_correct['rejects_at_0_01'], (
    f'Correct sampler rejected at p<0.01 (p={res_correct["p_value"]:.4f}) — sampler is WRONG'
)
print('  Correct sampler passes chi2 (p >= 0.01)')

# Negative control: incorrect sampler MUST be rejected at p < 0.01
res_bad = SpeculativeSampler.run_chi2_test(
    n_symbols=3, seq_len=3, n_trials=100_000, seed=6304, correct=False
)
print(f'  Incorrect: p={res_bad["p_value"]:.4f}  chi2={res_bad["chi2_stat"]:.2f}')
assert res_bad['rejects_at_0_01'], (
    f'Negative control NOT rejected at p<0.01 (p={res_bad["p_value"]:.4f}) — test is too weak'
)
print('  Negative control rejected (p < 0.01)')

print('\nPASS: T3 chi2 Markov test + negative control')


## Smoke test summary

If all cells printed `PASS`, the pipeline is healthy and you can proceed to full experiments.

**Next steps:**
1. Run `python scripts/setup_data.py --public` (CPU) to prepare dataset manifests on Drive.
2. Run `python scripts/setup_data.py --prefix-banks` (GPU) to prepare teacher and original-draft banks.
3. Generate the `exact_spec_original_gamma4` bank using `task3_specdec/speculative.py`.
4. Run full Task 1–3 experiments using the notebooks/scripts in each task directory.
